# Import Required Libraries
Use pandas for data manipulation and JSON to serialize image arrays.

In [1]:
import csv
import json
from pathlib import Path
import pandas as pd
import sqlite3

# Load CSV Data
Read the enriched_comments.csv file into a pandas DataFrame using the Python engine so multiline quoted comments are handled correctly.

In [2]:
csv_path = Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\public\enriched_comments.csv')
df = pd.read_csv(
    csv_path,
    encoding='utf-8',
    dtype=str,
    keep_default_na=False,
    quoting=csv.QUOTE_ALL,
    engine='python'
)
print('Loaded rows:', len(df))
print(df.columns.tolist())
df.head()

Loaded rows: 2542
['_id', 'comment', 'images', 'score', 'publish_date', 'room_type', 'fuzzy_room_type', 'travel_type', 'comment_len', 'log_comment_len', 'useful_count', 'log_useful_count', 'review_count', 'log_review_count', 'quality_score', 'categories']


,_id,comment,images,score,publish_date,room_type,fuzzy_room_type,travel_type,comment_len,log_comment_len,useful_count,log_useful_count,review_count,log_review_count,quality_score,categories
0,68027895e3c98b0941765706,房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一样 有些害怕...,"[ ""https://dimg04.c-ctrip.com/images/0230y1200...",5.0,2025-04-05,红棉大床套房,套房,家庭亲子,320,5.771441123130016,0,0.0,7,2.079441541679836,9,"['整体满意度', '餐饮设施', '前台服务']"
1,68027895e3c98b0941765707,花园酒店广州的老牌五星，一直期望的入住。房间我入住时要求前台帮我升级了新装修的房间，粤韵。其...,"[ ""https://dimg04.c-ctrip.com/images/023461200...",4.2,2025-04-07,红棉大床套房,套房,独自旅行,223,5.41164605185504,0,0.0,28,3.367295829986474,8,"['房间设施', '整体满意度', '交通便利性']"
2,68027895e3c98b0941765708,一进酒店就感觉来到了桃花源，迎宾人员都很热情，彬彬有礼，房间也非常不错，床品都很舒服，洗嗽用...,"[ ""https://dimg04.c-ctrip.com/images/023011200...",5.0,2025-04-15,花园双床房,双床房,商务出差,124,4.8283137373023015,0,0.0,11,2.4849066497880004,8,"['整体满意度', '前台服务', '餐饮设施']"
3,68027895e3c98b0941765709,总体来说很好。早餐有两个地方可以用，最上层的旋转餐厅用早餐感觉还是非常不错的。另外酒店的后花...,"[ ""https://dimg04.c-ctrip.com/images/0231r1200...",5.0,2025-04-15,花园大床房,大床房,情侣出游,99,4.605170185988092,1,0.6931471805599453,18,2.9444389791664403,9,"['餐饮设施', '公共设施', '整体满意度']"
4,68027895e3c98b094176570a,多年没来广州，临时起意过来住花园酒店，环境舒适，没有想象中的老式，全员服务在线，礼貌周到，给...,"[ ""https://dimg04.c-ctrip.com/images/0231o1200...",5.0,2025-04-06,花园双床房,双床房,家庭亲子,165,5.111987788356544,0,0.0,13,2.6390573296152584,9,"['整体满意度', '前台服务', '客房服务']"


# Filter Real Comments
Select rows where the comment field contains actual text.

In [3]:
real_df = df[df['comment'].astype(str).str.strip() != ''].copy()
print('Filtered real comments:', len(real_df))
real_df = real_df.reset_index(drop=True)
real_df.head()

Filtered real comments: 2542


,_id,comment,images,score,publish_date,room_type,fuzzy_room_type,travel_type,comment_len,log_comment_len,useful_count,log_useful_count,review_count,log_review_count,quality_score,categories
0,68027895e3c98b0941765706,房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一样 有些害怕...,"[ ""https://dimg04.c-ctrip.com/images/0230y1200...",5.0,2025-04-05,红棉大床套房,套房,家庭亲子,320,5.771441123130016,0,0.0,7,2.079441541679836,9,"['整体满意度', '餐饮设施', '前台服务']"
1,68027895e3c98b0941765707,花园酒店广州的老牌五星，一直期望的入住。房间我入住时要求前台帮我升级了新装修的房间，粤韵。其...,"[ ""https://dimg04.c-ctrip.com/images/023461200...",4.2,2025-04-07,红棉大床套房,套房,独自旅行,223,5.41164605185504,0,0.0,28,3.367295829986474,8,"['房间设施', '整体满意度', '交通便利性']"
2,68027895e3c98b0941765708,一进酒店就感觉来到了桃花源，迎宾人员都很热情，彬彬有礼，房间也非常不错，床品都很舒服，洗嗽用...,"[ ""https://dimg04.c-ctrip.com/images/023011200...",5.0,2025-04-15,花园双床房,双床房,商务出差,124,4.8283137373023015,0,0.0,11,2.4849066497880004,8,"['整体满意度', '前台服务', '餐饮设施']"
3,68027895e3c98b0941765709,总体来说很好。早餐有两个地方可以用，最上层的旋转餐厅用早餐感觉还是非常不错的。另外酒店的后花...,"[ ""https://dimg04.c-ctrip.com/images/0231r1200...",5.0,2025-04-15,花园大床房,大床房,情侣出游,99,4.605170185988092,1,0.6931471805599453,18,2.9444389791664403,9,"['餐饮设施', '公共设施', '整体满意度']"
4,68027895e3c98b094176570a,多年没来广州，临时起意过来住花园酒店，环境舒适，没有想象中的老式，全员服务在线，礼貌周到，给...,"[ ""https://dimg04.c-ctrip.com/images/0231o1200...",5.0,2025-04-06,花园双床房,双床房,家庭亲子,165,5.111987788356544,0,0.0,13,2.6390573296152584,9,"['整体满意度', '前台服务', '客房服务']"


# Connect to Database
Create a local SQLite preview database to verify the schema and import logic.

In [4]:
preview_db = Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\comments_import_preview.db')
conn = sqlite3.connect(preview_db)
create_sql = '''
CREATE TABLE IF NOT EXISTS comments (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  hotel_name TEXT NOT NULL,
  location TEXT NOT NULL,
  reviewer_name TEXT NOT NULL,
  review_title TEXT NOT NULL,
  review_text TEXT NOT NULL,
  review_date DATE NOT NULL,
  rating INTEGER NOT NULL,
  category TEXT NOT NULL,
  images TEXT NOT NULL
);'''
conn.execute(create_sql)
conn.commit()
print('Preview database ready at', preview_db)

Preview database ready at d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\comments_import_preview.db


# Insert Data into Table
Generate INSERT statements for the comments table and write them to SQL for backend execution.

In [5]:
def make_review_title(text: str) -> str:
    value = text.strip()
    title = value.split('\n', 1)[0][:40]
    return title or '用户评论'

output_path = Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\import_comments_real.sql')
output_path.parent.mkdir(parents=True, exist_ok=True)

statements = []
for _, row in real_df.iterrows():
    comment = str(row.get('comment', '')).strip().replace("'", "''")
    title = make_review_title(str(row.get('comment', '')).strip()).replace("'", "''")
    
    images = [img for img in str(row.get('images', '')).split('|') if img.strip()]
    images_json = json.dumps(images, ensure_ascii=False)
    images_escaped = images_json.replace("'", "''")
    
    publish_date = str(row.get('publish_date', '')).strip() or '1970-01-01'
    
    try:
        score = int(float(str(row.get('score', '0')).strip() or 0))
    except ValueError:
        score = 0
    
    category = str(row.get('categories', '') or row.get('travel_type', '') or '其他').strip().replace("'", "''")
    
    sql_stmt = (
        "INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, rating, category, images) VALUES ("
        f"'未知酒店', '未知地点', '匿名', '{title}', '{comment}', '{publish_date}', {score}, '{category}', '{images_escaped}');"
    )
    statements.append(sql_stmt)

with output_path.open('w', encoding='utf-8') as f:
    f.write('\n'.join(statements))

print(f'Written {len(statements)} INSERT statements to {output_path}')

Written 2542 INSERT statements to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\import_comments_real.sql


# Import to InsForge

Execute SQL statements in batches of 500 to avoid exceeding size limits.


In [6]:
# Read the generated SQL file
with open(output_path, 'r', encoding='utf-8') as f:
    all_statements = f.read()

# Split by ';' followed by a newline and 'INSERT'
# Each statement ends with ');' and the next starts with 'INSERT'
import re

# Split on the pattern: );  followed by newline(s) and INSERT
statements_list = re.split(r'\);\s*\n(?=INSERT)', all_statements)
# Add back the ); to each statement except the last
statements_list = [s.strip() + ');' if not s.strip().endswith(');') else s.strip() + ';' 
                   for s in statements_list if s.strip()]

print(f'Total statements: {len(statements_list)}')
print(f'Statement 1 preview: {statements_list[0][:200]}...')
print(f'Statement 2 preview: {statements_list[1][:200]}...')


Total statements: 2542
Statement 1 preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, rating, category, images) VALUES ('未知酒店', '未知地点', '匿名', '房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一', '...
Statement 2 preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, rating, category, images) VALUES ('未知酒店', '未知地点', '匿名', '花园酒店广州的老牌五星，一直期望的入住。房间我入住时要求前台帮我升级了新装修的房', '...


In [7]:
# Create batches of 500 statements
batch_size = 500
batches = []
for i in range(0, len(statements_list), batch_size):
    batch_stmts = statements_list[i:i+batch_size]
    # Combine with semicolon separator (each statement already has ;)
    batch_sql = ' '.join(batch_stmts)
    batches.append(batch_sql)

print(f'Total batches: {len(batches)}')
for idx, batch in enumerate(batches):
    print(f'Batch {idx+1}: {len(batch)} characters, preview: {batch[:100]}...')


Total batches: 6
Batch 1: 369049 characters, preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, r...
Batch 2: 349186 characters, preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, r...
Batch 3: 349258 characters, preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, r...
Batch 4: 335929 characters, preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, r...
Batch 5: 324001 characters, preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, r...
Batch 6: 25218 characters, preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, r...


In [8]:
# Export batches to file for manual execution
batches_dir = Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches')
batches_dir.mkdir(parents=True, exist_ok=True)

for idx, batch in enumerate(batches):
    batch_file = batches_dir / f'batch_{idx+1:02d}.sql'
    with open(batch_file, 'w', encoding='utf-8') as f:
        f.write(batch)
    print(f'Saved batch {idx+1} to {batch_file}')

print(f'\nTotal batches ready for execution: {len(batches)}')
print(f'To execute: Use mcp_insforge_run-raw-sql with each batch file')


Saved batch 1 to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches\batch_01.sql
Saved batch 2 to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches\batch_02.sql
Saved batch 3 to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches\batch_03.sql
Saved batch 4 to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches\batch_04.sql
Saved batch 5 to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches\batch_05.sql
Saved batch 6 to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches\batch_06.sql

Total batches ready for execution: 6
To execute: Use mcp_insforge_run-raw-sql with each batch file


In [9]:
# Split the statement list into smaller batches of 100 statements each
small_batches_dir = Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches_small')
small_batches_dir.mkdir(parents=True, exist_ok=True)

small_batch_size = 100
small_batches = []
for i in range(0, len(statements_list), small_batch_size):
    batch_stmts = statements_list[i:i+small_batch_size]
    batch_sql = ' '.join(batch_stmts)
    small_batches.append(batch_sql)
    batch_file = small_batches_dir / f'batch_small_{i//small_batch_size+1:03d}.sql'
    batch_file.write_text(batch_sql, encoding='utf-8')

print(f'Created {len(small_batches)} small batches in {small_batches_dir}')
for batch_file in sorted(small_batches_dir.glob('batch_small_*.sql')):
    print(batch_file.name, batch_file.stat().st_size)


Created 26 small batches in d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches_small
batch_small_001.sql 139545
batch_small_002.sql 105345
batch_small_003.sql 103593
batch_small_004.sql 106817
batch_small_005.sql 99168
batch_small_006.sql 97703
batch_small_007.sql 89484
batch_small_008.sql 105312
batch_small_009.sql 113924
batch_small_010.sql 103339
batch_small_011.sql 96571
batch_small_012.sql 100984
batch_small_013.sql 108784
batch_small_014.sql 105358
batch_small_015.sql 97079
batch_small_016.sql 95939
batch_small_017.sql 97298
batch_small_018.sql 107055
batch_small_019.sql 87166
batch_small_020.sql 95047
batch_small_021.sql 96855
batch_small_022.sql 89855
batch_small_023.sql 99754
batch_small_024.sql 93987
batch_small_025.sql 84620
batch_small_026.sql 36196


In [10]:
# Read the first batch SQL and prepare for execution
batch_files = sorted(Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches_small').glob('batch_small_*.sql'))
print(f'Found {len(batch_files)} batch files')

# Read first batch as example
first_batch_sql = batch_files[0].read_text(encoding='utf-8')
print(f'Batch 1 size: {len(first_batch_sql)} chars')
print(f'Batch 1 preview: {first_batch_sql[:200]}...')

# Prepare all batch SQL statements
all_batches_sql = {}
for batch_file in batch_files:
    batch_num = batch_file.stem.split('_')[-1]
    sql = batch_file.read_text(encoding='utf-8')
    all_batches_sql[batch_num] = sql
    
print(f'\nLoaded {len(all_batches_sql)} batches into memory for execution')


Found 26 batch files
Batch 1 size: 87010 chars
Batch 1 preview: INSERT INTO comments (hotel_name, location, reviewer_name, review_title, review_text, review_date, rating, category, images) VALUES ('未知酒店', '未知地点', '匿名', '房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一', '...

Loaded 26 batches into memory for execution


In [12]:
import subprocess
import json

# Save all batches to a JSON file for execution by a separate Python script
batches_data = {
    'api_key': 'ik_66c7a6a004185527ab44b13a721992b2',
    'base_url': 'https://6v7dmzzu.ap-southeast.insforge.app',
    'batches': all_batches_sql
}

batches_json = Path(r'd:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches.json')
with open(batches_json, 'w', encoding='utf-8') as f:
    json.dump(batches_data, f, ensure_ascii=False, indent=2)

print(f'Saved batches to {batches_json}')
print(f'Total batches: {len(all_batches_sql)}')

# Execute each batch
from urllib import request, error
import time

api_key = 'ik_66c7a6a004185527ab44b13a721992b2'
execute_url = 'https://6v7dmzzu.ap-southeast.insforge.app/rest/v1/query'
headers = {
    'Content-Type': 'application/json',
    'apikey': api_key,
    'Authorization': f'Bearer {api_key}'
}

success_count = 0
fail_count = 0
failed_batches = []

for batch_num_str in sorted(all_batches_sql.keys(), key=int):
    batch_sql = all_batches_sql[batch_num_str]
    
    # Send SQL query
    data = json.dumps({'query': batch_sql}).encode('utf-8')
    req = request.Request(execute_url, data=data, headers=headers, method='POST')
    
    try:
        with request.urlopen(req, timeout=30) as resp:
            status = resp.status
            body = resp.read().decode('utf-8')
        print(f'Batch {batch_num_str}: ✓ {status}')
        success_count += 1
        time.sleep(0.5)  # Rate limiting
    except error.HTTPError as e:
        print(f'Batch {batch_num_str}: ✗ {e.code} {e.reason}')
        fail_count += 1
        failed_batches.append(batch_num_str)
    except Exception as e:
        print(f'Batch {batch_num_str}: ✗ {e}')
        fail_count += 1
        failed_batches.append(batch_num_str)

print(f'\n✓ Successful: {success_count}')
print(f'✗ Failed: {fail_count}')
if failed_batches:
    print(f'Failed batches: {failed_batches}')


Saved batches to d:\复旦FDU\DSBA\大模型技术\Homework\Lab2\hotel-review\scripts\db\batches.json
Total batches: 26
Batch 001: ✗ 404 Not Found
Batch 002: ✗ 404 Not Found
Batch 003: ✗ 404 Not Found
Batch 004: ✗ 404 Not Found
Batch 005: ✗ 404 Not Found
Batch 006: ✗ 404 Not Found
Batch 007: ✗ 404 Not Found
Batch 008: ✗ 404 Not Found
Batch 009: ✗ 404 Not Found
Batch 010: ✗ 404 Not Found
Batch 011: ✗ 404 Not Found
Batch 012: ✗ 404 Not Found
Batch 013: ✗ 404 Not Found
Batch 014: ✗ 404 Not Found
Batch 015: ✗ 404 Not Found
Batch 016: ✗ 404 Not Found
Batch 017: ✗ 404 Not Found
Batch 018: ✗ 404 Not Found
Batch 019: ✗ 404 Not Found
Batch 020: ✗ 404 Not Found
Batch 021: ✗ 404 Not Found
Batch 022: ✗ 404 Not Found
Batch 023: ✗ 404 Not Found
Batch 024: ✗ The read operation timed out
Batch 025: ✗ 404 Not Found
Batch 026: ✗ 404 Not Found

✓ Successful: 0
✗ Failed: 26
Failed batches: ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019